In [13]:
import pandas as pd
import numpy as np

In [14]:
df_train = pd.read_csv("../../data/processed/train_engineered.csv")
df_val   = pd.read_csv("../../data/processed/val_engineered.csv")
df_test  = pd.read_csv("../../data/processed/test_engineered.csv")

In [4]:
import joblib
FEATURE_COLS = joblib.load("../../data/processed/feature_cols.pkl")
print(f"Loaded {len(FEATURE_COLS)} features")

Loaded 443 features


In [5]:
X_train = df_train[FEATURE_COLS].fillna(-999)
y_train = df_train['isFraud']

X_val = df_val[FEATURE_COLS].fillna(-999)
y_val = df_val['isFraud']

X_test = df_test[FEATURE_COLS].fillna(-999)
y_test = df_test['isFraud']

In [17]:
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("http://127.0.0.1:5000")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(roc_auc_score,average_precision_score,precision_recall_curve,classification_report)
import joblib


In [22]:
mlflow.set_experiment("fraud-detection-ieee-cis")
with mlflow.start_run(run_name="random_forest_classifier_reduced_features"):
    mlflow.log_param("model","RandomForest")
    mlflow.log_param("n_estimators",200)
    mlflow.log_param("class_weight","balanced")
    mlflow.log_param("feature_count",len(FEATURE_COLS))

    rf=RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=35,n_jobs=-1)
    rf.fit(X_train,y_train)
    y_prob_val_rf=rf.predict_proba(X_val)[:,1]
    pr_auc_rf=average_precision_score(y_val,y_prob_val_rf)
    roc_auc_rf=roc_auc_score(y_val,y_prob_val_rf)
    mlflow.log_metric("val_pr_auc",pr_auc_rf)
    mlflow.log_metric("val_roc_auc",roc_auc_rf)
    mlflow.sklearn.log_model(rf,'random_forest')
    print(f"RF Baseline — PR-AUC: {pr_auc_rf:.4f} | ROC-AUC: {roc_auc_rf:.4f}")

import joblib, os
os.makedirs('../../models', exist_ok=True)

joblib.dump(rf, '../../models/fraud_model.pkl')
joblib.dump(FEATURE_COLS, '../../models/feature_cols.pkl')
joblib.dump(y_prob_val_rf, '../../models/y_prob_val_rf.pkl')  # need this for threshold tuning
joblib.dump(y_val, '../../models/y_val.pkl')                  # need this too
joblib.dump(X_test, '../../models/X_test.pkl')                # for final eval later
print("Saved.")



2026/06/02 23:02:48 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-ieee-cis' does not exist. Creating a new experiment.


2026/06/02 23:04:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 23:04:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF Baseline — PR-AUC: 0.5531 | ROC-AUC: 0.9068
🏃 View run random_forest_classifier_reduced_features at: http://127.0.0.1:5000/#/experiments/1/runs/4597b43bd09b4b2aa4e3163d5d87d1f2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Saved.


In [15]:
from xgboost import XGBClassifier

In [23]:

n_neg=(y_train==0).sum()
n_pos=(y_train==1).sum()
spw=n_neg/n_pos
print(f"scale_pos_weight: {spw:.1f}")
with mlflow.start_run(run_name="xgboost_weighted_reduced_features"):
    mlflow.log_param("model","XGBoost")
    mlflow.log_param("scale_pos_weight",round(spw,1))
    mlflow.log_param("n_estimators",500)
    mlflow.log_param("max_depth",6)
    mlflow.log_param("learning_rate",0.05)
    mlflow.log_param("subsample",0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("feature_count",len(FEATURE_COLS))
    xgb = XGBClassifier(n_estimators=500,scale_pos_weight=spw,max_depth=6,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.8,random_state=35,eval_metric='aucpr',early_stopping_rounds=30,verbosity=0,tree_method='hist',n_jobs=-1
    )
    xgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
    y_prob_val_xgb=xgb.predict_proba(X_val)[:, 1]
    pr_auc_xgb=average_precision_score(y_val, y_prob_val_xgb)
    roc_auc_xgb=roc_auc_score(y_val, y_prob_val_xgb)

    mlflow.log_metric("val_pr_auc",  pr_auc_xgb)
    mlflow.log_metric("val_roc_auc", roc_auc_xgb)
    mlflow.sklearn.log_model(xgb, "xgboost")

    print(f"XGB — PR-AUC: {pr_auc_xgb:.4f} | ROC-AUC: {roc_auc_xgb:.4f}")

    



scale_pos_weight: 27.5
[0]	validation_0-aucpr:0.29029
[100]	validation_0-aucpr:0.45684
[200]	validation_0-aucpr:0.48452
[300]	validation_0-aucpr:0.49666
[400]	validation_0-aucpr:0.50728
[499]	validation_0-aucpr:0.51399


2026/06/02 23:18:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/02 23:18:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGB — PR-AUC: 0.5141 | ROC-AUC: 0.8578
🏃 View run xgboost_weighted_reduced_features at: http://127.0.0.1:5000/#/experiments/1/runs/6667f036e5c34ebea2c09658bf647fd1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [19]:
import joblib, os
os.makedirs('../../models', exist_ok=True)

joblib.dump(rf, '../../models/fraud_model.pkl')
joblib.dump(FEATURE_COLS, '../../models/feature_cols.pkl')

print("Saved.")

Saved.
